# RAG - Retrieval Augmented Generation
### required python library : which we use create a RAG pipeline
- **langchain** -> framework for LLM + RAG pipelines
- **pypdf, pymupdf** -> read PDFs
- **sentence-transformers** -> convert text -> embeddings (vectors)
- **chromadb** -> vector database

In [2]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [3]:
from langchain_core.documents import Document

In [4]:
sample_doc = Document(
    page_content="Hello World!",
    metadata={"source": "https://www.google.com"}
)

In [5]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

In [6]:
type(sample_doc)

langchain_core.documents.base.Document

In [7]:
# TExt data
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("data/python.txt", encoding="utf-8")

In [8]:
document = loader.load()

In [9]:
# PDF data
from langchain_community.document_loaders.pdf import PyPDFLoader

pdf_loader = PyPDFLoader("data/research2.pdf")

document = pdf_loader.load()

## Ingestion Pipeline

In [10]:
# Data => Documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [11]:
def load_all_pdf():
    folder_path = "data"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load() # list of documents (pages)

            all_docs.extend(doc) # add all pages into one list
            num_docs += 1

    print("total docs: ", num_docs)
    print("total_pages: ", len(all_docs))
    return all_docs

In [12]:
all_pdf_documents = load_all_pdf()

total docs:  2
total_pages:  32


In [13]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

### Chunks - Split Documents

In [14]:
# !pip install langchain_text_splitters

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_doc(documents, chunk_size=500, chunk_overlap=50):

    # chunk_size -> max charcters per chunk
    # chunk_overlap -> overlap between chunks (prevent lossing context)
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    # Big docs -> small chunks
    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [16]:
chunks = split_doc(all_pdf_documents)

In [17]:
len(chunks)

321

### Embedding

we use [**SentenceTransformer/`all-MiniLM-L6-v2`**](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2#all-minilm-l6-v2) 

In [18]:
from sentence_transformers import SentenceTransformer

In [19]:
# we create class : for reusability
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):  # load embedding model

        self.model_name=model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name) # load pretrained embedding model
        print("embedding dimensions=",self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self, text): # do embedding for given data
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embedding shape: ", embeddings.shape)
        return embeddings

In [20]:
# initialize Embeddding
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


C:\Users\ranja\anaconda3\envs\ml\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


embedding dimensions= 384


## Vector Store
### Vector Database - (ChromaDB)

In [21]:
import chromadb
import uuid # helps to create indexes

**VectorStoreManager** = Database layer for our AI
Its help us:
- to creating a **ChromaDB** vector database
- Storing: chunks, embeddings, metadata

> PDF -> Vector DB -> Searchable by Ai

In [22]:
# controle all db operations in one place 
class VectorStoreManager:
    # persist_directory -> where DB store
    # collection_name -> table name
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.persist_directory = persist_directory # where DB store
        self.collection_name = collection_name # table name
        self.client = None # connection to db
        self.collection = None # specific table

        self._initialize_store() # auto db set when object is created

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        
        # create DB client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create/get the collection : (collection exists- load ELSE create new one) 
        # creating a collection (table) named "pdf_documents"
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embedding in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings): # helps to prevents data mismatch
            raise ValueError("num of docs does not match num of emeddings")

        # storage list => ids, embedding, documents, metadata
        ids = [] # unique indetifiers
        all_metadata = [] # extra info
        documents_content = [] # actual text
        embeddings_list = [] # vector data

        # zip(documents, embeddings) -> combine both lists pair by pair
        # enumerate -> add index to each pair : ex -[ (0, (doc1, vec1)), ...]
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}" # generate unique id
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist()) # ChromaDB expects python list

        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embeddings_list
        )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())
        

In [23]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 322


In [24]:
# data => documents => chunks => embedddings => store in vector store

texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

embedding shape:  (321, 384)
total documents added in vector store= 321
docs in collection: 643


## Retrieval Pipeline

#### `RAGRetriever` = Search Engine of AI
> Query -> Embedding -> Vector DB Search -> Filter -> Return

In [26]:
from sklearn.metrics.pairwise import cosine_similarity
# used to compare vectors

In [27]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    # query -> user question
    # top_k -> return top k results
    # score_threshold -> filter low-quality matches
    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query embedding (query => vector)
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0] # from list of vectors -> single vector

        # semantic search : database return top k values based on query_embeddings
        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()], # .tolist() -> convert Numpy -python list
            n_results = top_k
        )

        # chromaDB returns:
        # results = {
        #     "ids": [[...]],
        #     "documents": [[...]],
        #     "metadatas": [[...]],
        #     "distances": [[...]]
        # }
        # dobule list because: quey multiples input at once

        # prepare output list
        retrieved_docs=[] # context (final result container)
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance
                # distance(0.1) => good
                # distance(0.8) => bad

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i+1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")
            
        else:
            print("no documents found")

        return retrieved_docs
        

In [28]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [29]:
rag_retriever.retrieve("explain RAG and its work flow")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape:  (1, 384)
retrieved 5 documents


[{'id': 'doc_7ed65097-93a3-47cc-b8ae-e7082a319d74',
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'metadata': {'keywords': '',
   'doc_index': 88,
   'title': '',
   'page_label': '1',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'trapped': '/False',
   'author': '',
   'content_length': 288,
   'creator': 'LaTeX with hyperref',
   'creationdate': '2024-03-28T00:54:45+00:00',
   'moddate': '2024-03-28T00:54:45+00:00',
   'producer': 'pdfTeX-1.40.25',
   'subject': '',
   'page': 0,
   'total_pages': 21,
   'source': 'data\\research2.pdf'},
  'distance': 0.6116979718208313,
  'similarity_score': 0.3883020281791687,
  'rank': 1},
 {'id': 'doc_b3f7ac12-9dc1-